# Regenerate all caches: TFM PCJ pipeline

Cada celda es **independiente**. Para regenerar UN solo modulo, ejecuta solo
esa celda (asume que las dependencias upstream existen en cache).

Para regenerar **TODO desde 0**, ejecuta de arriba a abajo con `FORCE = True`
en cada celda (default).

## Flags por celda
- `FORCE=True` borra los caches del modulo y los regenera.
- `RETRAIN=True` re-entrena modelos pesados (M04 SVI, M05 LightGBM, M08 CatBoost,
  M11 SVI). Default `True`. Ponlo a `False` para reusar el modelo cacheado y
  solo regenerar las aggs derivadas.
- `SKIP_AUDIT=True` salta el audit raw (celda 1). Default `True` (ya validado).

## Orden topologico
M03 → M04 → M05 → M06 → M07 → M08 → M09 → M10 → M11 → smoke final

M10 es la mas cara (~30-60 min, OBSO/PPCF 25Hz x 64 matches).


In [1]:
# === [0] Setup ==========================================================
from pathlib import Path
import sys, time, shutil, warnings, os
import polars as pl

# --- Filtros globales de warnings ----------------------------------------
# Estos warnings vienen de librerias externas (socceraction, jax, numpy en
# operaciones arctan/divide-by-zero defensivamente manejadas, pandas
# chained-assignment en codigo de socceraction). Son ruido conocido — no
# bugs nuestros — y estaban poluyendo la salida del NB.
warnings.filterwarnings('ignore', category=UserWarning,   module='socceraction')
warnings.filterwarnings('ignore', category=FutureWarning, module='socceraction')
warnings.filterwarnings('ignore', category=DeprecationWarning, module='socceraction')
warnings.filterwarnings('ignore', category=FutureWarning, module='pandas')
warnings.filterwarnings('ignore', category=RuntimeWarning, message='invalid value')
warnings.filterwarnings('ignore', category=RuntimeWarning, message='divide by zero')
warnings.filterwarnings('ignore', category=RuntimeWarning, message='overflow')
warnings.filterwarnings('ignore', category=UserWarning,   module='lightgbm')
warnings.filterwarnings('ignore', category=UserWarning,
                        message='X does not have valid feature names')  # sklearn vs LightGBM
warnings.filterwarnings('ignore', category=DeprecationWarning, module='jax')
warnings.filterwarnings('ignore', category=FutureWarning, module='jax')

# Silencia loggers verbose
os.environ['JAX_PLATFORMS'] = os.environ.get('JAX_PLATFORMS', 'cpu')
import logging
for log_name in ('numexpr', 'numpyro', 'numba', 'matplotlib'):
    logging.getLogger(log_name).setLevel(logging.WARNING)

# --- Localiza REPO (CWD = repo o notebooks/) -----------------------------
candidates = [Path('.').resolve(), Path('..').resolve()]
REPO = next((p for p in candidates
             if (p / 'src' / 'M01_loader_pff.py').exists()), None)
assert REPO is not None, f'No encuentro REPO desde {Path.cwd()}'
SRC = REPO / 'src'
DERIVED = REPO / 'data' / 'parquet' / 'derived'
sys.path.insert(0, str(SRC))

print(f'REPO   : {REPO}')
print(f'DERIVED: {DERIVED}')
print(f'CWD    : {Path.cwd()}')


def _clean_dir(path: Path):
    """Borra dir o fichero si existe (no falla si no existe)."""
    if not path.exists():
        return
    if path.is_dir():
        shutil.rmtree(path)
    else:
        path.unlink()


def _clean_files(*paths: Path):
    """Borra ficheros individuales si existen."""
    for p in paths:
        if p.exists():
            p.unlink()


REPO   : /home/joriol/TFM
DERIVED: /home/joriol/TFM/data/parquet/derived
CWD    : /home/joriol/TFM/notebooks


In [2]:
# === [1] Audit raw lossless (PFF + SB + Wyscout) ========================
# Verifica round-trip lossless de los parquets raw vs JSON original.
# Solo verificacion, no regenera nada.
SKIP_AUDIT = True  # cambia a False para ejecutar (~3-5 min)

if SKIP_AUDIT:
    print('  AUDIT SKIPPED (SKIP_AUDIT=True). Cambia a False para ejecutar.')
else:
    from extract.audit import run_full_audit
    t0 = time.time()
    res = run_full_audit()
    print(f'  audit completo en {time.time()-t0:.1f}s, all_ok={res["all_ok"]}')


  AUDIT SKIPPED (SKIP_AUDIT=True). Cambia a False para ejecutar.


In [3]:
# === [2] M03 preprocess - enrich_events =================================
# Outputs : data/parquet/derived/preprocess/events_enriched/{match_id}.parquet x 64
# Consumes: M01 (PFF events + metadata) + M02 (SB matches + events para
#           goals_timeline con sgc real PFF, X4 fix)
# Re-run downstream: ninguno (events_enriched no se consume por otros modulos)
FORCE = True

import M03_preprocess as m03

target_dir = DERIVED / 'preprocess' / 'events_enriched'
if FORCE:
    print(f'  borrando {target_dir}')
    _clean_dir(target_dir)

t0 = time.time()
out = m03.cache_all_enriched(overwrite=FORCE)
print(f'  enrich_events: {len(out)} matches en {time.time()-t0:.1f}s')

# Sanity
sample_path = next(target_dir.glob('*.parquet'))
sample = pl.read_parquet(sample_path)
print(f'  sample {sample_path.name}: {sample.height:,} rows x {sample.width} cols')


  borrando /home/joriol/TFM/data/parquet/derived/preprocess/events_enriched
  enrich_events: 64 matches en 21.3s
  sample 3823.parquet: 2,202 rows x 36 cols


In [4]:
# === [3] M04 WP - training matrix + SVI fit + temperature + per_minute ===
# Outputs : derived/wp/training/training_matrix.parquet
#           derived/wp/model/wp_regulation.pkl
#           derived/wp/per_minute.parquet
# Consumes: Wyscout events, SB events (no WC22), PFF events para apply
# Re-run downstream: ninguno actualmente (M12+ futuros consumiran)
FORCE = True
RETRAIN = True   # re-entrenar SVI bayes (~3-5 min) + temperature scaling

import M04_wp as m04

if FORCE:
    if RETRAIN:
        _clean_files(m04._CACHE / 'training_matrix.parquet',
                     m04._MODEL / 'wp_regulation.pkl')
    _clean_files(m04._DERIVED / 'per_minute.parquet')

t0 = time.time()
X, info = m04.build_training_matrix(cache=True)
print(f'  training matrix: {info}')

fit_path = m04._MODEL / 'wp_regulation.pkl'
if fit_path.exists() and not RETRAIN:
    fit = m04.load_fit(fit_path)
    print('  modelo cargado desde cache (RETRAIN=False)')
else:
    X_tr, X_val = m04.train_val_split(X, val_frac=0.2, seed=42)
    fit = m04.fit_wp(X_tr, n_steps=3000)
    X_calib = m04.build_wc22_groups_calib_matrix()
    T_opt = m04.fit_temperature(fit, X_calib)
    fit['temperature'] = T_opt
    m04.save_fit(fit)
    print(f'  modelo entrenado, T optimo = {T_opt:.4f}')

group_ctx = m04.build_wc22_group_context()
out = m04.cache_all_wp(fit, overwrite=True, group_ctx=group_ctx)
big = pl.read_parquet(out)
print(f'  per_minute: {big.height:,} filas, '
      f'{big["match_id"].n_unique()}/64 matches en {time.time()-t0:.0f}s')


  training matrix: {'n_samples': 186930, 'n_matches': 2077, 'cached': False}
  step     0  elbo_loss=1030093.06
  step   500  elbo_loss=125076.92
  step  1000  elbo_loss=119532.23
  step  1500  elbo_loss=118105.45
  step  2000  elbo_loss=117619.98
  step  2500  elbo_loss=117362.12
  modelo entrenado, T optimo = 1.0313
  per_minute: 5,910 filas, 64/64 matches en 743s


In [5]:
# === [4] M05 PSxG - Optuna 60 + LightGBM + isotonic + apply WC22 ========
# Outputs : derived/psxg/training_shots.parquet, wc22_shots.parquet
#           derived/psxg/model/psxg_lgb.pkl
#           derived/psxg/shots.parquet (con sb_match_id, no match_id ambiguo)
# Consumes: SB events + 360 freeze (Euro20+Euro24+Bundes23 train, WC22 apply)
# Re-run downstream: M06 nearmiss, M10 (xg_grid usa training_shots)
FORCE = True
RETRAIN = True   # LightGBM con Optuna 60 trials (~10-15 min)

import M05_psxg as m05

if FORCE:
    if RETRAIN:
        _clean_files(m05._DERIVED / 'training_shots.parquet',
                     m05._DERIVED / 'wc22_shots.parquet',
                     m05._MODEL / 'psxg_lgb.pkl')
    _clean_files(m05._DERIVED / 'shots.parquet')

t0 = time.time()
train = m05.build_training_shots(cache=True)
print(f'  training shots: {train.height} ({int(train["_label"].sum())} goles)')

fit_path = m05._MODEL / 'psxg_lgb.pkl'
if fit_path.exists() and not RETRAIN:
    fit = m05.load_fit(fit_path)
    print('  modelo cargado desde cache (RETRAIN=False)')
else:
    fit = m05.fit_psxg(train, n_folds=5, seed=42, n_trials=60)
    m05.save_fit(fit)
    m = fit['metrics']
    print(f'  modelo entrenado, AUC OOF cal {m["auc_psxg_calibrated"]:.4f} '
          f'vs SB baseline {m["auc_baseline_sb_xg"]:.4f}')

out = m05.cache_wc22_psxg(fit, overwrite=True)
big = pl.read_parquet(out)
print(f'  shots WC22: {big.height} ({big["is_goal"].sum()} goles) '
      f'en {time.time()-t0:.0f}s')
print(f'  cols: {big.columns}')


  training shots: 3545 (391 goles)
Optuna tuning (n_trials=60, 5-fold CV by match)...
  best AUC tuning: 0.9751
  best params: {'n_estimators': 731, 'max_depth': 6, 'learning_rate': 0.0746436649382348, 'num_leaves': 99, 'min_child_samples': 17, 'subsample': 0.8358922978496754, 'colsample_bytree': 0.9267859424872934, 'reg_alpha': 0.07474548753984195, 'reg_lambda': 0.9061423008852212, 'min_split_gain': 0.07558390547622877}
  modelo entrenado, AUC OOF cal 0.9740 vs SB baseline 0.8265
  shots WC22: 1494 (195 goles) en 58s
  cols: ['sb_match_id', 'event_uuid', 'period', 'minute', 'second', 'shot_outcome', 'is_goal', 'xg_baseline', 'psxg']


In [6]:
# === [5] M06 nearmiss - 5 tipos near-miss desde PSxG WC22 ===============
# Outputs : derived/nearmiss/nearmiss_table.parquet (con sb_match_id)
# Consumes: M05 shots.parquet, SB events + 360 freeze
# Re-run downstream: M13 AIPW (futuro)
FORCE = True

import M06_nearmiss as m06

if FORCE:
    _clean_files(m06._DERIVED / 'nearmiss_table.parquet')

t0 = time.time()
nm = m06.build_near_miss_table(cache=True, overwrite=True)
print(f'  near-miss: {nm.height} en {time.time()-t0:.0f}s')
print(m06.summary_by_type(nm))
print(f'  cols: {nm.columns}')


  near-miss: 57 en 4s
shape: (4, 2)
┌───────────────────────┬─────┐
│ near_miss_type        ┆ len │
│ ---                   ┆ --- │
│ str                   ┆ u32 │
╞═══════════════════════╪═════╡
│ a_woodwork            ┆ 12  │
│ b_offside_close       ┆ 5   │
│ c_save_psxg           ┆ 38  │
│ d_goal_line_clearance ┆ 2   │
└───────────────────────┴─────┘
  cols: ['sb_match_id', 'event_uuid', 'period', 'minute', 'second', 'team_id', 'team_name', 'shot_outcome', 'is_goal', 'xg_baseline', 'psxg', 'near_miss_type', 'margin_info']


In [7]:
# === [6] M07 shocks - 172 shocks-gol + ventanas +-10min por jugador ====
# Outputs : derived/shocks/shocks_table.parquet (t_event_seconds en sgc real PFF)
# Consumes: M03 goals_timeline (X4 fix), player_minutes
# Re-run downstream: M08, M09, M10, M11 per_shock_window
FORCE = True

import M07_shocks as m07

if FORCE:
    _clean_files(m07._DERIVED / 'shocks_table.parquet')

t0 = time.time()
sh = m07.build_shocks_table(cache=True, overwrite=True)
print(f'  shocks: {sh.height:,} rows ({sh["shock_id"].n_unique()} shocks unicos) '
      f'en {time.time()-t0:.0f}s')
print(m07.summary_shocks(sh))


  shocks: 3,788 rows (172 shocks unicos) en 7s
shape: (2, 8)
┌──────────────┬────────┬──────────┬─────────────┬──────────────┬───────────┬───────┬──────┐
│ shock_type   ┆ n_rows ┆ n_shocks ┆ n_trunc_pre ┆ n_trunc_post ┆ n_overlap ┆ n_sub ┆ n_et │
│ ---          ┆ ---    ┆ ---      ┆ ---         ┆ ---          ┆ ---       ┆ ---   ┆ ---  │
│ str          ┆ u32    ┆ u32      ┆ u32         ┆ u32          ┆ u32       ┆ u32   ┆ u32  │
╞══════════════╪════════╪══════════╪═════════════╪══════════════╪═══════════╪═══════╪══════╡
│ GOAL_AGAINST ┆ 1893   ┆ 172      ┆ 375         ┆ 330          ┆ 947       ┆ 345   ┆ 44   │
│ GOAL_FOR     ┆ 1895   ┆ 172      ┆ 374         ┆ 330          ┆ 946       ┆ 334   ┆ 44   │
└──────────────┴────────┴──────────┴─────────────┴──────────────┴───────────┴───────┴──────┘


In [8]:
# === [7] M08 ataque - atomic SPADL + CatBoost VAEP + apply + aggs ======
# Outputs : derived/ataque/training_atomic.parquet, wc22_atomic.parquet
#           derived/ataque/model/vaep_atk_*.cbm + meta.pkl
#           derived/ataque/sb_to_pff_player_map.parquet
#           derived/ataque/per_minute.parquet (X3: ambos ids + sec_abs)
#           derived/ataque/per_shock_window.parquet (period filter contra contam.)
# Consumes: SB events socceraction, M07 shocks
# Re-run downstream: M09, M10, M11 (player_map), M12 DiD futuro
FORCE = True
RETRAIN = True   # CatBoost + Optuna 30 trials (~30-45 min)

import M08_ataque as atk

if FORCE:
    if RETRAIN:
        _clean_files(atk._DERIVED / 'training_atomic.parquet',
                     atk._DERIVED / 'wc22_atomic.parquet')
        _clean_dir(atk._MODEL_DIR)
        # Z01 cachea features atomic-VAEP por (provider, game_id) en
        # REPO/cache/vaep/. Si reentrenamos M08 con atomic actions
        # potencialmente nuevos, ese cache puede quedar stale.
        _clean_dir(REPO / 'cache' / 'vaep')
    _clean_files(atk._DERIVED / 'sb_to_pff_player_map.parquet',
                 atk._DERIVED / 'per_minute.parquet',
                 atk._DERIVED / 'per_shock_window.parquet')

t0 = time.time()
print('[1] atomic SPADL training (Euro20+Euro24+Bundes23)...')
train_df = atk.build_training_atomic(overwrite=RETRAIN)
print(f'  training atomic: {len(train_df):,} acciones')

print('[2] atomic SPADL WC22...')
wc22_df = atk.build_wc22_atomic(overwrite=RETRAIN)
print(f'  wc22 atomic: {len(wc22_df):,} acciones')

print('[3] VAEP model...')
fit_path = atk._MODEL_DIR / 'vaep_atk_meta.pkl'
if fit_path.exists() and not RETRAIN:
    fit = atk.load_models()
    print('  modelo cargado desde cache (RETRAIN=False)')
else:
    fit = atk.train_vaep_model(train_df, n_folds=5, seed=42,
                                tune=True, n_trials=30)
    atk.save_models(fit)
    m = fit['metrics']
    print(f'  modelo entrenado, AUC scores OOF cal '
          f'{m["auc_scores_oof_cal"]:.4f}')

print('[4] aplicando VAEP a WC22...')
wc22_with_vaep = atk.apply_vaep_to_wc22(fit, wc22_df)

print('[5] sb -> pff player map...')
mapping = atk.build_sb_to_pff_player_map(cache=True)
mapped = mapping.filter(pl.col('pff_player_id').is_not_null()).height
print(f'  mapping: {mapped}/{mapping.height} matched '
      f'({100*mapped/mapping.height:.1f}%)')

print('[6] per_minute...')
per_min = atk.aggregate_per_player_minute(wc22_with_vaep, cache=True)
print(f'  per_minute: {per_min.height:,} filas')
print(f'  cols: {per_min.columns}')

print('[7] per_shock_window...')
per_shock = atk.aggregate_per_shock_window(per_min, mapping, cache=True)
print(f'  per_shock: {per_shock.height:,} filas')
print(f'  cols: {per_shock.columns}')
print(f'  total: {time.time()-t0:.0f}s')


[1] atomic SPADL training (Euro20+Euro24+Bundes23)...
  [Euro20] 51 partidos...
  [Euro24] 51 partidos...
  [Bundes23] 34 partidos...
  training atomic: 453,930 acciones
[2] atomic SPADL WC22...
  [WC22] 64 partidos...
  wc22 atomic: 209,225 acciones
[3] VAEP model...
  Optuna tuning (30 trials, 3-fold, scores objective)...
  best hparams: {'depth': 4, 'learning_rate': 0.19523070091617523, 'l2_leaf_reg': 19.64406549768874, 'bagging_temperature': 0.7546466097724522}
  5-fold CV by match (scores + concedes)...
    fold 1/5 done
    fold 2/5 done
    fold 3/5 done
    fold 4/5 done
    fold 5/5 done
  fitting FINAL models on all training...
  modelo entrenado, AUC scores OOF cal 0.8310
[4] aplicando VAEP a WC22...
[5] sb -> pff player map...
  mapping: 509/680 matched (74.9%)
[6] per_minute...
  per_minute: 57,119 filas
  cols: ['pff_match_id', 'sb_match_id', 'pff_player_id', 'sb_player_id', 'period', 'minute_in_period', 'sec_abs', 'score_atk_minute', 'vaep_minute', 'n_actions']
[7] per_s

In [9]:
# === [8] M09 defensa - VDEP-like + def_third + pressing intensity ======
# Outputs : derived/defensa/def_third_context.parquet (tracking PFF 25Hz)
#           derived/defensa/per_minute.parquet (con vdep_like_minute)
#           derived/defensa/per_shock_window.parquet
# Consumes: M08 model + atomic + player_map; M07 shocks; tracking PFF 25Hz
# Re-run downstream: M12 DiD futuro
FORCE = True

import M09_defensa as df9

if FORCE:
    _clean_files(df9._DERIVED / 'def_third_context.parquet',
                 df9._DERIVED / 'per_minute.parquet',
                 df9._DERIVED / 'per_shock_window.parquet')

t0 = time.time()
print('[1] def_third + pressing context (tracking 25Hz x 64 matches)...')
ctx = df9.build_def_third_all(cache=True)
print(f'  context: {ctx.height:,} rows')

print('[2] per_minute (joining VAEP defensive_value + context)...')
per_min = df9.aggregate_per_player_minute(cache=True)
print(f'  per_minute: {per_min.height:,} filas')
print(f'  cols: {per_min.columns}')

print('[3] per_shock_window...')
per_shock = df9.aggregate_per_shock_window(cache=True)
print(f'  per_shock: {per_shock.height:,} filas')
print(f'  cols: {per_shock.columns}')
print(f'  total: {time.time()-t0:.0f}s')


[1] def_third + pressing context (tracking 25Hz x 64 matches)...
  10/64 en 8.6s
  20/64 en 15.2s
  30/64 en 21.6s
  40/64 en 28.1s
  50/64 en 34.5s
  60/64 en 41.1s
  context: 126,930 rows
[2] per_minute (joining VAEP defensive_value + context)...
  per_minute: 57,119 filas
  cols: ['pff_match_id', 'sb_match_id', 'pff_player_id', 'sb_player_id', 'period', 'minute_in_period', 'sec_abs', 'score_def_minute', 'vdep_like_minute', 'n_def_actions', 'n_actions_total', 'def_third_pct', 'press_intensity_frames', 'oppo_possession_frames']
[3] per_shock_window...
  per_shock: 3,788 filas
  cols: ['pff_match_id', 'sb_match_id', 'shock_id', 'shock_type', 'pff_player_id', 'sb_player_id', 'score_def_pre', 'score_def_post', 'vdep_like_pre', 'vdep_like_post', 'n_def_actions_pre', 'n_def_actions_post', 'press_frames_pre', 'press_frames_post']
  total: 47s


In [10]:
# === [9] M10 offball - OBSO Spearman 2018 + C-OBSO + xG grid ===========
# Outputs : derived/offball/xg_grid.npy
#           derived/offball/per_minute.parquet (period+minute_in_period+sec_abs)
#           derived/offball/per_shock_window.parquet
# Consumes: M05 training_shots (xG grid), tracking PFF 25Hz, M08 player_map
# Re-run downstream: M12 DiD futuro
# CARO: ~30-60 min para los 64 matches a 25Hz full
FORCE = True

import M10_offball as m10

if FORCE:
    _clean_files(m10._DERIVED / 'xg_grid.npy',
                 m10._DERIVED / 'per_minute.parquet',
                 m10._DERIVED / 'per_shock_window.parquet')

t0 = time.time()
print('[1] xG grid (de M05 training shots)...')
xg = m10.build_xg_grid(cache=True)
print(f'  xg_grid {xg.shape}: min={xg.min():.3f}, max={xg.max():.3f}')

print('[2] OBSO + C-OBSO (PPCF Z02 25Hz x 64 matches)...')
print('  ESTO TARDA 30-60 MIN — paciencia')
per_min = m10.aggregate_per_player_minute(cache=True)
print(f'  per_minute: {per_min.height:,} filas')
print(f'  cols: {per_min.columns}')

print('[3] per_shock_window...')
per_shock = m10.aggregate_per_shock_window(cache=True)
print(f'  per_shock: {per_shock.height:,} filas')
print(f'  cols: {per_shock.columns}')
print(f'  total: {time.time()-t0:.0f}s')


[1] xG grid (de M05 training shots)...
  xg_grid (10, 7): min=0.005, max=0.161
[2] OBSO + C-OBSO (PPCF Z02 25Hz x 64 matches)...
  ESTO TARDA 30-60 MIN — paciencia
  1/64 en 359s (last match 359s)
  2/64 en 766s (last match 407s)
  3/64 en 1163s (last match 397s)
  4/64 en 1543s (last match 380s)
  5/64 en 1888s (last match 346s)
  6/64 en 2358s (last match 469s)
  7/64 en 2797s (last match 439s)
  8/64 en 3186s (last match 389s)
  9/64 en 3488s (last match 302s)
  10/64 en 3725s (last match 237s)
  11/64 en 3998s (last match 273s)
  12/64 en 4234s (last match 236s)
  13/64 en 4502s (last match 268s)
  14/64 en 4725s (last match 223s)
  15/64 en 4963s (last match 238s)
  16/64 en 5189s (last match 226s)
  17/64 en 5382s (last match 193s)
  18/64 en 5620s (last match 238s)
  19/64 en 5828s (last match 208s)
  20/64 en 6113s (last match 285s)
  21/64 en 6295s (last match 182s)
  22/64 en 6529s (last match 234s)
  23/64 en 6849s (last match 320s)
  24/64 en 7094s (last match 245s)
  25/64

In [11]:
# === [10] M11 fisico - Bradley 2024 metrics + state-space bayes residual ==
# Outputs : derived/fisico/raw_per_minute.parquet
#           derived/fisico/model/phys_state.pkl
#           derived/fisico/per_minute.parquet (con sec_abs)
#           derived/fisico/per_shock_window.parquet
# Consumes: tracking PFF 25Hz, M07 shocks, M08 player_map
# Re-run downstream: M12 DiD futuro
FORCE = True
RETRAIN = True   # SVI multivariate 4000 steps (~3-5 min)

import M11_fisico as m11

if FORCE:
    if RETRAIN:
        _clean_files(m11._DERIVED / 'model' / 'phys_state.pkl')
    _clean_files(m11._DERIVED / 'raw_per_minute.parquet',
                 m11._DERIVED / 'per_minute.parquet',
                 m11._DERIVED / 'per_shock_window.parquet')

t0 = time.time()
print('[1] raw metrics Bradley 2024 (Butterworth + Hampel + segmentation)...')
raw = m11.build_raw_per_minute(cache=True, overwrite=FORCE)
print(f'  raw: {raw.height:,} filas')
print(f'  cols: {raw.columns}')

print('[2] SVI multivariate bayes + score_phys (residual z-score)...')
pm = m11.cache_score_phys(overwrite=FORCE, n_steps=4000)
print(f'  per_minute: {pm.height:,} filas, '
      f'score_phys mean={pm["score_phys"].mean():+.4f} '
      f'std={pm["score_phys"].std():.4f}')

print('[3] per_shock_window...')
per_shock = m11.aggregate_per_shock_window(cache=True)
print(f'  per_shock: {per_shock.height:,} filas')
print(f'  cols: {per_shock.columns}')
print(f'  total: {time.time()-t0:.0f}s')


[1] raw metrics Bradley 2024 (Butterworth + Hampel + segmentation)...
  10/64 en 24.3s
  20/64 en 46.7s
  30/64 en 70.5s
  40/64 en 93.7s
  50/64 en 117.4s
  60/64 en 140.6s
  raw: 146,942 filas
  cols: ['minute_in_period', 'distance_m', 'hsr_s', 'sprint_s', 'sprint_count', 'psv95', 'n_high_accel', 'n_high_decel', 'z1_m', 'z2_m', 'z3_m', 'z4_m', 'z5_m', 'hmld_m', 'n_frames', 'pff_match_id', 'pff_player_id', 'period']
[2] SVI multivariate bayes + score_phys (residual z-score)...
  fitting SVI multivariate phys model (146,942 obs, 4000 steps, 3 targets)...
  step     0  elbo_loss=12021142.00
  step   500  elbo_loss=329950.50
  step  1000  elbo_loss=290976.78
  step  1500  elbo_loss=291797.56
  step  2000  elbo_loss=289976.38
  step  2500  elbo_loss=289597.59
  step  3000  elbo_loss=290404.62
  step  3500  elbo_loss=288293.03
  per_minute: 145,351 filas, score_phys mean=+0.0029 std=0.8517
[3] per_shock_window...
  per_shock: 3,788 filas
  cols: ['pff_match_id', 'sb_match_id', 'shock_id', 

In [12]:
# === [11] Smoke final — listado + tamano de todos los caches ============
def _walk_size(base: Path):
    if not base.exists():
        return None, []
    files = []
    for ext in ('*.parquet', '*.pkl', '*.cbm', '*.npy'):
        files.extend(base.rglob(ext))
    files = sorted(set(files))
    return sum(f.stat().st_size for f in files) / 1024 / 1024, files

print(f'{"PATH":<70} {"SIZE_MB":>10}')
print('-' * 85)
total = 0.0
for sub in ['preprocess/events_enriched', 'wp', 'psxg', 'nearmiss',
            'shocks', 'ataque', 'defensa', 'offball', 'fisico']:
    base = DERIVED / sub
    size, files = _walk_size(base)
    if size is None:
        print(f'{str(sub):<70} {"":>10}  MISSING')
        continue
    total += size
    print(f'{str(sub):<70} {size:>10.2f}')
    for f in files:
        fsize = f.stat().st_size / 1024 / 1024
        rel = f.relative_to(DERIVED)
        print(f'  {str(rel):<68} {fsize:>10.2f}')
print('-' * 85)
print(f'{"TOTAL":<70} {total:>10.2f} MB')


PATH                                                                      SIZE_MB
-------------------------------------------------------------------------------------
preprocess/events_enriched                                                  73.02
  preprocess/events_enriched/10502.parquet                                   1.26
  preprocess/events_enriched/10503.parquet                                   1.36
  preprocess/events_enriched/10504.parquet                                   1.07
  preprocess/events_enriched/10505.parquet                                   1.00
  preprocess/events_enriched/10506.parquet                                   1.57
  preprocess/events_enriched/10507.parquet                                   1.33
  preprocess/events_enriched/10508.parquet                                   1.65
  preprocess/events_enriched/10509.parquet                                   0.98
  preprocess/events_enriched/10510.parquet                                   1.71
  preprocess